In [20]:
import torch
import torchvision
import torch.optim as optim
import torch.nn as nn

In [21]:
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform = torchvision.transforms.ToTensor())
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
images, labels = next(iter(trainloader))

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=torchvision.transforms.ToTensor())
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

In [22]:
class cnn(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.features = torch.nn.Sequential(
        torch.nn.Conv2d(3, 32, kernel_size=3, padding=1),
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(kernel_size=2, stride=2),

        torch.nn.Conv2d(32, 64, kernel_size=3, padding=1),
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(kernel_size=2, stride=2),

        torch.nn.Conv2d(64, 64, kernel_size=3, padding=1),
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(kernel_size=2, stride=2)
    )
    self.classifier = torch.nn.Sequential(
      torch.nn.Flatten(),
      torch.nn.Linear(64 * 4 * 4, 64),
      torch.nn.ReLU(),
      torch.nn.Linear(64, 10)
      )

  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return x

model = cnn()

In [24]:
model = cnn()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)
epochs = 20
for epoch in range(epochs):
    totalloss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()

        totalloss += loss.item()

    avg_loss = totalloss / len(trainloader)
    print("epoch- ", epoch, "loss- ", avg_loss)

epoch-  0 loss-  2.299079834652679
epoch-  1 loss-  2.242693540080429
epoch-  2 loss-  2.03696023808111
epoch-  3 loss-  1.9131489493657865
epoch-  4 loss-  1.782911515144436
epoch-  5 loss-  1.6510233670244436
epoch-  6 loss-  1.5642105029976887
epoch-  7 loss-  1.505221056816218
epoch-  8 loss-  1.4626688274276225
epoch-  9 loss-  1.4179657850119158
epoch-  10 loss-  1.3731167284424042
epoch-  11 loss-  1.3279529029451063
epoch-  12 loss-  1.2805487907603574
epoch-  13 loss-  1.2382973152048447
epoch-  14 loss-  1.1967496810209415
epoch-  15 loss-  1.1584218610125734
epoch-  16 loss-  1.1230574465163834
epoch-  17 loss-  1.0848630283342298
epoch-  18 loss-  1.0571370780315545
epoch-  19 loss-  1.0266481586124585


In [25]:
correct = 0
total = 0

with torch.no_grad():
  for images, labels in testloader:
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 56.84
